# Why Refractive Calibration Matters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tlancaster6/AquaCal/blob/main/docs/tutorials/02_synthetic_validation.ipynb)

Underwater imaging systems view targets through a flat air-water interface. A standard
pinhole camera model ignores refraction at this interface, treating light as travelling
in a straight line from camera to target. AquaCal corrects this using Snell's law.

**But does it actually matter?** This tutorial answers that question with two
controlled synthetic experiments:

1. **Parameter Fidelity** — Can both models fit the data? Which one recovers correct parameters?
2. **Depth Generalization** — When we calibrate at one depth and measure at another, how do the models compare?

Because we control all ground truth parameters, we can measure exactly where each
model fails and why.

In [ ]:
# Colab setup — clones the repo and installs AquaCal if running on Google Colab.
# This notebook imports test helpers from the repo's tests/ directory, so a full
# clone is needed (not just pip install). When running locally, this cell is a no-op.
import sys

if "google.colab" in sys.modules:
    import os
    if not os.path.exists("/content/AquaCal/src"):
        print("Colab detected — cloning repo and installing AquaCal...")
        !git clone --depth 1 https://github.com/tlancaster6/AquaCal.git /content/AquaCal
        !pip install -q -e /content/AquaCal
    else:
        print("AquaCal repo already present.")
    os.chdir("/content/AquaCal")
    print("Done.")
else:
    print("Local environment — skipping install.")

In [ ]:
# Toggle rig size:
#   "small" — 4 cameras, 20 frames, ideal conditions. Fast, ~2 min total.
#   "large" — 13 cameras, 30 frames, realistic noise. Compelling results, ~60 min total.
#   "calibration" — Use geometry from a real calibration.json file.
RIG_SIZE = "large"

# Only used when RIG_SIZE = "calibration":
CALIBRATION_PATH = r"C:\Users\tucke\Desktop\Aqua\AquaCal\release_calibration\calibration.json"

OUTPUT_DIR = Path("output")  # exported data artifacts

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make sure project root is on path (needed when running as a notebook)
_project_root = Path().resolve()
while _project_root.name and not (_project_root / "src").exists():
    _project_root = _project_root.parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from aquacal.config.schema import BoardConfig, InterfaceParams
from aquacal.core.board import BoardGeometry
from aquacal.validation.reprojection import compute_reprojection_errors
from tests.synthetic.experiment_helpers import (
    calibrate_synthetic,
    compute_per_camera_errors,
    evaluate_reconstruction,
)
from tests.synthetic.ground_truth import (
    SyntheticScenario,
    create_scenario,
    generate_dense_xy_grid,
    generate_real_rig_array,
    generate_real_rig_trajectory,
    generate_synthetic_detections,
    rig_from_calibration,
)

# Consistent color palette for all plots
COLOR_REFRACTIVE = "#2196F3"      # Blue
COLOR_NON_REFRACTIVE = "#F44336"  # Red
LABEL_REFRACTIVE = "Refractive (AquaCal)"
LABEL_NON_REFRACTIVE = "Non-refractive (pinhole)"

print("Imports complete.")

## Preset Configuration

The `"small"` preset uses the `"ideal"` scenario: 4 cameras arranged in a grid,
near the water surface (15 cm above), boards at 0.25�0.45 m depth, no noise.
This is fast and lets you verify the math.

The `"large"` preset uses the `"realistic"` scenario: 13 cameras matching real
hardware (center + two concentric rings), 0.75 m above water, boards at 0.9�1.5 m
depth, 0.5 px noise. This produces compelling quantitative results.

The `"calibration"` option loads camera geometry from an existing calibration.json
file, letting you run the same experiments with your real rig layout.

In [ ]:
if RIG_SIZE == "small":
    # create_scenario("ideal"): 4 cameras at Z=0, water at Z=0.15, boards at Z=0.25-0.45m
    # All test/sweep depths must be > 0.15m (below water surface)
    SCENARIO_NAME = "ideal"
    # Exp2: calibrate on narrow band, test at these depths (all underwater)
    EXP2_CALIB_RANGE = (0.30, 0.40)
    EXP2_TEST_DEPTHS = [0.22, 0.28, 0.35, 0.42, 0.50]
    EXP2_N_CALIB_FRAMES = 15
elif RIG_SIZE == "calibration":
    _cal_intr, _cal_extr, _cal_wz, _cal_board = rig_from_calibration(CALIBRATION_PATH)
    SCENARIO_NAME = "calibration"
    # Derive experiment parameters from the loaded rig geometry
    _water_z_mean = sum(_cal_wz.values()) / len(_cal_wz)
    EXP2_CALIB_RANGE = (_water_z_mean + 0.20, _water_z_mean + 0.40)
    EXP2_TEST_DEPTHS = [
        round(_water_z_mean + offset, 2)
        for offset in [0.05, 0.15, 0.30, 0.45, 0.60, 0.80, 1.10, 1.50]
    ]
    EXP2_N_CALIB_FRAMES = 50
    print(f"Loaded {len(_cal_intr)} cameras from {Path(CALIBRATION_PATH).name}")
    print(f"  Mean water_z: {_water_z_mean:.3f} m")
else:
    # create_scenario("realistic"): 13 cameras, water at ~0.75m, boards at 0.9-1.5m depth
    SCENARIO_NAME = "realistic"
    # Exp2: calibrate on narrow band, test across a wide depth range
    EXP2_CALIB_RANGE = (0.95, 1.05)
    EXP2_TEST_DEPTHS = [0.80, 0.90, 1.00, 1.10, 1.20, 1.40, 1.70, 2.00]
    EXP2_N_CALIB_FRAMES = 50

print(f"RIG_SIZE = {RIG_SIZE!r}")
print(f"  Scenario:            {SCENARIO_NAME}")
print(f"  Exp2 calib range:    {EXP2_CALIB_RANGE}")
print(f"  Exp2 test depths:    {EXP2_TEST_DEPTHS}")

## Experiment 1: Parameter Fidelity

**Question:** When we calibrate both models on identical data, do they recover
the correct camera parameters?

We generate a synthetic scenario with known ground truth, then run the full
calibration pipeline twice — once with the refractive model (n_water = 1.333)
and once with the non-refractive pinhole approximation (n_water = 1.0). Both
optimizations minimize reprojection error on the same 2D observations.

The key insight: **a model can fit 2D observations without learning correct 3D geometry**.
If the model is wrong, it will compensate by adjusting focal length and camera positions.

In [ ]:
print("=== Experiment 1: Parameter Fidelity ===")
if SCENARIO_NAME == "calibration":
    _cal_intr, _cal_extr, _cal_wz, _cal_board = rig_from_calibration(CALIBRATION_PATH)
    _cal_poses = generate_real_rig_trajectory(
        n_frames=30,
        depth_range=(EXP2_CALIB_RANGE[0], EXP2_TEST_DEPTHS[-1]),
        seed=42,
    )
    scenario = SyntheticScenario(
        name="calibration",
        board_config=_cal_board,
        intrinsics=_cal_intr,
        extrinsics=_cal_extr,
        water_zs=_cal_wz,
        board_poses=_cal_poses,
        noise_std=0.5,
        description=f"Real rig from {Path(CALIBRATION_PATH).name}: {len(_cal_intr)} cameras",
    )
else:
    scenario = create_scenario(SCENARIO_NAME, seed=42)
print(f"Scenario: {scenario.description}")
print(f"  Cameras: {len(scenario.intrinsics)}")
print(f"  Frames:  {len(scenario.board_poses)}")
print(f"  Noise:   {scenario.noise_std} px")

print("\nCalibrating refractive model (n_water=1.333)...")
result_refr, _ = calibrate_synthetic(scenario, n_water=1.333, refine_intrinsics=True)
errors_refr = compute_per_camera_errors(result_refr, scenario)
print(f"  Reprojection RMS: {result_refr.diagnostics.reprojection_error_rms:.4f} px")

print("\nCalibrating non-refractive model (n_water=1.0)...")
result_nonrefr, _ = calibrate_synthetic(scenario, n_water=1.0, refine_intrinsics=True)
errors_nonrefr = compute_per_camera_errors(result_nonrefr, scenario)
print(f"  Reprojection RMS: {result_nonrefr.diagnostics.reprojection_error_rms:.4f} px")

print("\nBoth calibrations complete.")
print(f"  Note: similar reprojection errors ({result_refr.diagnostics.reprojection_error_rms:.3f} vs "
      f"{result_nonrefr.diagnostics.reprojection_error_rms:.3f} px) — "
      "look at parameter errors next.")

# Interface distance (water_z) recovery — refractive model only
gt_wz = np.mean(list(scenario.water_zs.values()))
est_wz = np.mean([c.water_z for c in result_refr.cameras.values()])
wz_err_mm = (est_wz - gt_wz) * 1000
wz_err_pct = (est_wz - gt_wz) / gt_wz * 100
print(f"\nInterface distance (water_z) recovery [refractive model only]:")
print(f"  Ground truth: {gt_wz * 1000:.1f} mm")
print(f"  Estimated:    {est_wz * 1000:.1f} mm")
print(f"  Error:        {wz_err_mm:+.2f} mm ({wz_err_pct:+.3f}%)")

### Focal Length Recovery

When refraction bends rays at the interface, a non-refractive model compensates by
adjusting focal length. The result: a systematically biased focal length that happens
to minimize 2D reprojection error but does not match the true optics.

In [ ]:
camera_names = sorted(scenario.intrinsics.keys())
x = np.arange(len(camera_names))
width = 0.35

focal_refr = [errors_refr[cam]["focal_length_error_pct"] for cam in camera_names]
focal_nonrefr = [errors_nonrefr[cam]["focal_length_error_pct"] for cam in camera_names]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, focal_refr, width, label=LABEL_REFRACTIVE, color=COLOR_REFRACTIVE)
ax.bar(x + width / 2, focal_nonrefr, width, label=LABEL_NON_REFRACTIVE, color=COLOR_NON_REFRACTIVE)
ax.set_xlabel("Camera")
ax.set_ylabel("Focal Length Error (%)")
ax.set_title("Focal Length Recovery Error vs Ground Truth")
ax.set_xticks(x)
ax.set_xticklabels(camera_names, rotation=45, ha="right")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

mean_focal_refr = np.mean([abs(e) for e in focal_refr])
mean_focal_nonrefr = np.mean([abs(e) for e in focal_nonrefr])
print(f"Mean absolute focal length error:")
print(f"  Refractive:     {mean_focal_refr:.3f}%")
print(f"  Non-refractive: {mean_focal_nonrefr:.3f}%")

### Camera Z Position Recovery

Along with focal length, the non-refractive model misplaces camera positions along
the Z axis (depth). These two biases together produce a model that fits 2D projections
but triangulates 3D positions incorrectly.

Because the reference camera (cam0) is fixed at the world-frame origin during
optimization, its raw Z error is always zero by construction. To reveal the
systematic shift the model would have applied to the entire rig, we subtract
the mean Z error of the free cameras from all cameras (mean-shift correction).

In [ ]:
z_refr_raw = [errors_refr[cam]["z_position_error_mm"] for cam in camera_names]
z_nonrefr_raw = [errors_nonrefr[cam]["z_position_error_mm"] for cam in camera_names]

# The reference camera (cam0) is pinned at ground truth during optimization,
# so its raw error is always 0. To reveal the systematic Z shift the model
# would have applied to cam0, subtract the mean error of the free cameras
# from all cameras. This gives each camera's estimated distance from its
# true position, including the reference.
ref_cam = camera_names[0]
free_cams = camera_names[1:]

mean_z_refr = np.mean([errors_refr[cam]["z_position_error_mm"] for cam in free_cams])
mean_z_nonrefr = np.mean([errors_nonrefr[cam]["z_position_error_mm"] for cam in free_cams])

z_refr = [v - mean_z_refr for v in z_refr_raw]
z_nonrefr = [v - mean_z_nonrefr for v in z_nonrefr_raw]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, z_refr, width, label=LABEL_REFRACTIVE, color=COLOR_REFRACTIVE)
ax.bar(x + width / 2, z_nonrefr, width, label=LABEL_NON_REFRACTIVE, color=COLOR_NON_REFRACTIVE)
ax.set_xlabel("Camera")
ax.set_ylabel("Z Position Error (mm)")
ax.set_title("Camera Z Position Recovery Error (mean-shift corrected)")
ax.set_xticks(x)
ax.set_xticklabels(camera_names, rotation=45, ha="right")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

mean_z_refr_abs = np.mean([abs(e) for e in z_refr])
mean_z_nonrefr_abs = np.mean([abs(e) for e in z_nonrefr])
print(f"Mean absolute Z position error (mean-shift corrected):")
print(f"  Refractive:     {mean_z_refr_abs:.2f} mm")
print(f"  Non-refractive: {mean_z_nonrefr_abs:.2f} mm")

### Camera XY Position Recovery

Top-down view of camera positions. Arrows show how far each estimated position
is from ground truth. A good calibration produces very short arrows.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for i, cam_name in enumerate(camera_names):
    C_gt = scenario.extrinsics[cam_name].C
    C_refr = result_refr.cameras[cam_name].extrinsics.C
    C_nonrefr = result_nonrefr.cameras[cam_name].extrinsics.C

    gt_label = "Ground Truth" if i == 0 else None
    refr_label = LABEL_REFRACTIVE if i == 0 else None
    nonrefr_label = LABEL_NON_REFRACTIVE if i == 0 else None

    ax.scatter(C_gt[0], C_gt[1], color="black", s=120, zorder=3, label=gt_label, marker="o")
    ax.scatter(C_refr[0], C_refr[1], color=COLOR_REFRACTIVE, s=60, zorder=4, label=refr_label, alpha=0.8)
    ax.scatter(C_nonrefr[0], C_nonrefr[1], color=COLOR_NON_REFRACTIVE, s=60, zorder=4, label=nonrefr_label, alpha=0.8)

    dx_refr = C_refr[0] - C_gt[0]
    dy_refr = C_refr[1] - C_gt[1]
    dx_nonrefr = C_nonrefr[0] - C_gt[0]
    dy_nonrefr = C_nonrefr[1] - C_gt[1]

    arrow_kw = dict(head_width=0.015, head_length=0.008, length_includes_head=True)
    ax.arrow(C_gt[0], C_gt[1], dx_refr, dy_refr, color=COLOR_REFRACTIVE, alpha=0.6, **arrow_kw)
    ax.arrow(C_gt[0], C_gt[1], dx_nonrefr, dy_nonrefr, color=COLOR_NON_REFRACTIVE, alpha=0.6, **arrow_kw)

ax.set_xlabel("X Position (m)")
ax.set_ylabel("Y Position (m)")
ax.set_title("Camera XY Position Recovery (arrows = estimation error)")
ax.set_aspect("equal")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

mean_xy_refr = np.mean([errors_refr[cam]["xy_position_error_mm"] for cam in camera_names])
mean_xy_nonrefr = np.mean([errors_nonrefr[cam]["xy_position_error_mm"] for cam in camera_names])
print(f"Mean XY position error:")
print(f"  Refractive:     {mean_xy_refr:.2f} mm")
print(f"  Non-refractive: {mean_xy_nonrefr:.2f} mm")

### Distortion Coefficient Recovery

The non-refractive model doesn't just bias focal length and position — it also
absorbs refraction into the radial distortion coefficients (k1, k2). Since
refraction produces a depth-dependent, spatially-varying ray bending that true
radial distortion cannot represent, the learned coefficients are physically wrong.

In [ ]:
k1_refr = [errors_refr[cam]["k1_error"] for cam in camera_names]
k1_nonrefr = [errors_nonrefr[cam]["k1_error"] for cam in camera_names]
k2_refr = [errors_refr[cam]["k2_error"] for cam in camera_names]
k2_nonrefr = [errors_nonrefr[cam]["k2_error"] for cam in camera_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

ax1.bar(x - width / 2, k1_refr, width, label=LABEL_REFRACTIVE, color=COLOR_REFRACTIVE)
ax1.bar(x + width / 2, k1_nonrefr, width, label=LABEL_NON_REFRACTIVE, color=COLOR_NON_REFRACTIVE)
ax1.set_xlabel("Camera")
ax1.set_ylabel("k1 Error (absolute)")
ax1.set_title("k1 Distortion Coefficient Error")
ax1.set_xticks(x)
ax1.set_xticklabels(camera_names, rotation=45, ha="right")
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

ax2.bar(x - width / 2, k2_refr, width, label=LABEL_REFRACTIVE, color=COLOR_REFRACTIVE)
ax2.bar(x + width / 2, k2_nonrefr, width, label=LABEL_NON_REFRACTIVE, color=COLOR_NON_REFRACTIVE)
ax2.set_xlabel("Camera")
ax2.set_ylabel("k2 Error (absolute)")
ax2.set_title("k2 Distortion Coefficient Error")
ax2.set_xticks(x)
ax2.set_xticklabels(camera_names, rotation=45, ha="right")
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()

print(f"Mean absolute distortion error:")
print(f"  k1 — Refractive: {np.mean([abs(e) for e in k1_refr]):.6f},  "
      f"Non-refractive: {np.mean([abs(e) for e in k1_nonrefr]):.6f}")
print(f"  k2 — Refractive: {np.mean([abs(e) for e in k2_refr]):.6f},  "
      f"Non-refractive: {np.mean([abs(e) for e in k2_nonrefr]):.6f}")

### Experiment 1 Takeaway

Both models achieve similar reprojection error — the 2D fitting quality looks the same.
But the non-refractive model absorbs the refraction effect into systematically biased
focal lengths and camera positions. The refractive model recovers the true physical
parameters.

**Why it matters:** Biased parameters produce systematic 3D reconstruction errors
that grow with depth — exactly what Experiment 2 will show.

## Experiment 2: Depth Generalization

**Question:** When we calibrate at one depth range and measure at another depth,
which model generalizes better?

Real underwater surveys rarely stay at exactly the calibration depth. Fish tracking
systems follow animals at varying depths; coral surveys cover different depth strata.

We calibrate both models on a narrow depth band, then evaluate 3D reconstruction
accuracy at a range of test depths — both inside and outside the calibration zone.
The refractive model should generalize because it learned the correct geometry;
the non-refractive model's bias is depth-dependent.

In [ ]:
print("=== Experiment 2: Depth Generalization ===")
print(f"Calibration depth range: {EXP2_CALIB_RANGE}")
print(f"Test depths: {EXP2_TEST_DEPTHS}")

# Build rig from the preset scenario's parameters
# For the small preset: use the ideal scenario's rig geometry
# For the large preset: use the realistic 13-camera rig
# For the calibration preset: use the loaded calibration geometry
if RIG_SIZE == "small":
    from tests.synthetic.ground_truth import create_scenario as _cs
    _base = _cs("ideal", seed=42)
    intrinsics_exp2 = _base.intrinsics
    extrinsics_exp2 = _base.extrinsics
    water_zs_exp2 = _base.water_zs
    board_config_exp2 = _base.board_config
    noise_std_exp2 = 0.0
    n_grid_exp2 = 4  # smaller grid for 4-camera rig
    xy_extent_exp2 = 0.05
    tilt_deg_exp2 = 2.0
elif RIG_SIZE == "calibration":
    intrinsics_exp2, extrinsics_exp2, water_zs_exp2, board_config_exp2 = rig_from_calibration(CALIBRATION_PATH)
    noise_std_exp2 = 0.5
    n_grid_exp2 = 7
    xy_extent_exp2 = 0.5
    tilt_deg_exp2 = 3.0
else:
    intrinsics_exp2, extrinsics_exp2, water_zs_exp2 = generate_real_rig_array(
        height_above_water=0.75, height_variation=0.002, seed=42
    )
    board_config_exp2 = BoardConfig(
        squares_x=12, squares_y=9,
        square_size=0.060, marker_size=0.045,
        dictionary="DICT_5X5_100",
    )
    noise_std_exp2 = 0.5
    n_grid_exp2 = 7
    xy_extent_exp2 = 0.5
    tilt_deg_exp2 = 3.0

board_exp2 = BoardGeometry(board_config_exp2)

# Build calibration scenario on narrow depth band
if RIG_SIZE == "small":
    from aquacal.datasets.synthetic import generate_board_trajectory
    camera_positions_exp2 = {cam: ext.C for cam, ext in extrinsics_exp2.items()}
    calib_poses_exp2 = generate_board_trajectory(
        n_frames=EXP2_N_CALIB_FRAMES,
        camera_positions=camera_positions_exp2,
        water_zs=water_zs_exp2,
        depth_range=EXP2_CALIB_RANGE,
        xy_extent=xy_extent_exp2,
        seed=42,
    )
else:
    calib_poses_exp2 = generate_real_rig_trajectory(
        n_frames=EXP2_N_CALIB_FRAMES,
        depth_range=EXP2_CALIB_RANGE,
        seed=42,
    )

calib_scenario_exp2 = SyntheticScenario(
    name="exp2_calib",
    board_config=board_config_exp2,
    intrinsics=intrinsics_exp2,
    extrinsics=extrinsics_exp2,
    water_zs=water_zs_exp2,
    board_poses=calib_poses_exp2,
    noise_std=noise_std_exp2,
    description=f"Calibration at {EXP2_CALIB_RANGE}",
)

print("Calibrating refractive model...")
result_refr_exp2, _ = calibrate_synthetic(calib_scenario_exp2, n_water=1.333, refine_intrinsics=True)
print(f"  RMS: {result_refr_exp2.diagnostics.reprojection_error_rms:.4f} px")

print("Calibrating non-refractive model...")
result_nonrefr_exp2, _ = calibrate_synthetic(calib_scenario_exp2, n_water=1.0, refine_intrinsics=True)
print(f"  RMS: {result_nonrefr_exp2.diagnostics.reprojection_error_rms:.4f} px")

print("\nBoth calibrations complete. Evaluating at test depths...")

In [ ]:
results_refr_exp2 = []
results_nonrefr_exp2 = []

for depth in EXP2_TEST_DEPTHS:
    print(f"  Evaluating Z = {depth:.2f} m...")

    test_poses = generate_dense_xy_grid(
        depth=depth,
        n_grid=n_grid_exp2,
        xy_extent=xy_extent_exp2,
        tilt_deg=tilt_deg_exp2,
        frame_offset=1000,
        seed=42 + int(depth * 100),
    )

    test_detections = generate_synthetic_detections(
        intrinsics=intrinsics_exp2,
        extrinsics=extrinsics_exp2,
        water_zs=water_zs_exp2,
        board=board_exp2,
        board_poses=test_poses,
        noise_std=noise_std_exp2,
        seed=42 + int(depth * 100),
    )

    err_refr = evaluate_reconstruction(result_refr_exp2, board_exp2, test_detections)
    err_nonrefr = evaluate_reconstruction(result_nonrefr_exp2, board_exp2, test_detections)

    # Reprojection error at this test depth using GT board poses
    reproj_refr = compute_reprojection_errors(result_refr_exp2, test_detections, test_poses)
    reproj_nonrefr = compute_reprojection_errors(result_nonrefr_exp2, test_detections, test_poses)

    true_dist = board_config_exp2.square_size
    scale_refr = 1.0 + (err_refr.signed_mean / true_dist)
    scale_nonrefr = 1.0 + (err_nonrefr.signed_mean / true_dist)

    results_refr_exp2.append({
        "depth": depth,
        "signed_mean_mm": err_refr.signed_mean * 1000,
        "rmse_mm": err_refr.rmse * 1000,
        "scale": scale_refr,
        "reproj_rms_px": reproj_refr.rms,
        "spatial": err_refr.spatial,
    })
    results_nonrefr_exp2.append({
        "depth": depth,
        "signed_mean_mm": err_nonrefr.signed_mean * 1000,
        "rmse_mm": err_nonrefr.rmse * 1000,
        "scale": scale_nonrefr,
        "reproj_rms_px": reproj_nonrefr.rms,
        "spatial": err_nonrefr.spatial,
    })

print("Depth sweep complete.")

### Signed Error vs Depth

The shaded region marks the calibration depth range. Inside that range, both models
fit well (they were trained on it). Outside, the non-refractive model's error grows
because the wrong geometry can't generalize to different depths.

In [ ]:
depths_plot = [r["depth"] for r in results_refr_exp2]
signed_refr = [r["signed_mean_mm"] for r in results_refr_exp2]
signed_nonrefr = [r["signed_mean_mm"] for r in results_nonrefr_exp2]

fig, ax = plt.subplots(figsize=(9, 5))
ax.axvspan(EXP2_CALIB_RANGE[0], EXP2_CALIB_RANGE[1], alpha=0.15, color="gray", label="Calibration Range")
ax.plot(depths_plot, signed_refr, marker="o", color=COLOR_REFRACTIVE, label=LABEL_REFRACTIVE, linewidth=2)
ax.plot(depths_plot, signed_nonrefr, marker="s", color=COLOR_NON_REFRACTIVE, label=LABEL_NON_REFRACTIVE, linewidth=2)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Test Depth (m)")
ax.set_ylabel("Signed Mean Error (mm)")
ax.set_title("Signed Error vs Test Depth (calibrated on narrow band)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
rmse_refr = [r["rmse_mm"] for r in results_refr_exp2]
rmse_nonrefr = [r["rmse_mm"] for r in results_nonrefr_exp2]

fig, ax = plt.subplots(figsize=(9, 5))
ax.axvspan(EXP2_CALIB_RANGE[0], EXP2_CALIB_RANGE[1], alpha=0.15, color="gray", label="Calibration Range")
ax.plot(depths_plot, rmse_refr, marker="o", color=COLOR_REFRACTIVE, label=LABEL_REFRACTIVE, linewidth=2)
ax.plot(depths_plot, rmse_nonrefr, marker="s", color=COLOR_NON_REFRACTIVE, label=LABEL_NON_REFRACTIVE, linewidth=2)
ax.set_xlabel("Test Depth (m)")
ax.set_ylabel("Reconstruction RMSE (mm)")
ax.set_title("Reconstruction RMSE vs Test Depth")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

max_rmse_refr = max(rmse_refr)
max_rmse_nonrefr = max(rmse_nonrefr)
print(f"Maximum RMSE across all test depths:")
print(f"  Refractive:     {max_rmse_refr:.2f} mm")
print(f"  Non-refractive: {max_rmse_nonrefr:.2f} mm")

### Reprojection Error vs Depth

Reprojection error measures how well the calibrated model explains the 2D observations
at each test depth using the ground-truth board poses. Unlike 3D reconstruction error,
this is a purely 2D metric — but it still degrades for the non-refractive model
because refraction bends rays by a depth-dependent amount that a fixed pinhole model
cannot accommodate.

In [ ]:
reproj_refr = [r["reproj_rms_px"] for r in results_refr_exp2]
reproj_nonrefr = [r["reproj_rms_px"] for r in results_nonrefr_exp2]

fig, ax = plt.subplots(figsize=(9, 5))
ax.axvspan(EXP2_CALIB_RANGE[0], EXP2_CALIB_RANGE[1], alpha=0.15, color="gray", label="Calibration Range")
ax.plot(depths_plot, reproj_refr, marker="o", color=COLOR_REFRACTIVE, label=LABEL_REFRACTIVE, linewidth=2)
ax.plot(depths_plot, reproj_nonrefr, marker="s", color=COLOR_NON_REFRACTIVE, label=LABEL_NON_REFRACTIVE, linewidth=2)
ax.set_xlabel("Test Depth (m)")
ax.set_ylabel("Reprojection RMS (px)")
ax.set_title("Reprojection Error vs Test Depth (using GT board poses)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

### Spatial Error Distribution

Are reconstruction errors uniform across the field of view, or concentrated in
certain regions? The plots below show signed reconstruction error at each measurement
position (XY) for the shallowest and deepest test depths. Each dot is a pairwise
distance measurement; color encodes signed error (blue = underestimate, red = overestimate).

At depths far from calibration, the non-refractive model's error is not only larger
but spatially structured — it varies across the field of view because refraction
angle depends on the incidence geometry, which a fixed pinhole distortion model
cannot capture.

In [ ]:
# Pick shallowest and deepest test depths for comparison
depth_indices = [0, len(EXP2_TEST_DEPTHS) - 1]
depth_labels = ["Shallowest", "Deepest"]

fig, axes = plt.subplots(len(depth_indices), 2, figsize=(14, 6 * len(depth_indices)))

# Compute shared color scale across all panels
all_errors_mm = []
for di in depth_indices:
    for results in [results_refr_exp2, results_nonrefr_exp2]:
        sp = results[di]["spatial"]
        if sp is not None:
            all_errors_mm.extend((sp.signed_errors * 1000).tolist())
if all_errors_mm:
    vmax = max(abs(min(all_errors_mm)), abs(max(all_errors_mm)))
else:
    vmax = 1.0

for row, (di, dlabel) in enumerate(zip(depth_indices, depth_labels)):
    depth_val = EXP2_TEST_DEPTHS[di]
    for col, (results, model_label) in enumerate([
        (results_refr_exp2, "Refractive"),
        (results_nonrefr_exp2, "Non-refractive"),
    ]):
        ax = axes[row, col]
        sp = results[di]["spatial"]
        if sp is not None and len(sp.signed_errors) > 0:
            sc = ax.scatter(
                sp.positions[:, 0], sp.positions[:, 1],
                c=sp.signed_errors * 1000,
                cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                s=15, alpha=0.7,
            )
            plt.colorbar(sc, ax=ax, label="Signed Error (mm)")
        ax.set_xlabel("X (m)")
        ax.set_ylabel("Y (m)")
        ax.set_title(f"{model_label} — {dlabel} ({depth_val:.2f} m)")
        ax.set_aspect("equal")
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
scale_refr = [r["scale"] for r in results_refr_exp2]
scale_nonrefr = [r["scale"] for r in results_nonrefr_exp2]

fig, ax = plt.subplots(figsize=(9, 5))
ax.axvspan(EXP2_CALIB_RANGE[0], EXP2_CALIB_RANGE[1], alpha=0.15, color="gray", label="Calibration Range")
ax.plot(depths_plot, scale_refr, marker="o", color=COLOR_REFRACTIVE, label=LABEL_REFRACTIVE, linewidth=2)
ax.plot(depths_plot, scale_nonrefr, marker="s", color=COLOR_NON_REFRACTIVE, label=LABEL_NON_REFRACTIVE, linewidth=2)
ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--", label="True scale (1.0)")
ax.set_xlabel("Test Depth (m)")
ax.set_ylabel("Scale Factor (measured / true)")
ax.set_title("Scale Factor vs Test Depth")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

### Exported Data

The two CSVs below capture the per-record data behind all plots in this notebook,
so downstream consumers can reproduce statistics and figures without running AquaCal.

In [ ]:
# --- Exp1: per-camera parameter errors ---
# Compute mean Z shift of free cameras for mean-shift correction
_free = [c for c in camera_names if c != camera_names[0]]
_mz = {
    "refractive": np.mean([errors_refr[c]["z_position_error_mm"] for c in _free]),
    "non_refractive": np.mean([errors_nonrefr[c]["z_position_error_mm"] for c in _free]),
}

rows_exp1 = []
for cam in camera_names:
    for label, errors, result in [
        ("refractive", errors_refr, result_refr),
        ("non_refractive", errors_nonrefr, result_nonrefr),
    ]:
        C_gt = scenario.extrinsics[cam].C
        C_est = result.cameras[cam].extrinsics.C
        rows_exp1.append({
            "camera": cam,
            "model": label,
            "focal_length_error_pct": errors[cam]["focal_length_error_pct"],
            "z_position_error_mm": errors[cam]["z_position_error_mm"] - _mz[label],
            "xy_position_error_mm": errors[cam]["xy_position_error_mm"],
            "gt_x_m": C_gt[0], "gt_y_m": C_gt[1], "gt_z_m": C_gt[2],
            "est_x_m": C_est[0], "est_y_m": C_est[1], "est_z_m": C_est[2],
            "reprojection_rms_px": result.diagnostics.reprojection_error_rms,
        })
df_exp1 = pd.DataFrame(rows_exp1)

# --- Exp2: depth-generalization metrics ---
rows_exp2 = []
for r_refr, r_nonrefr in zip(results_refr_exp2, results_nonrefr_exp2):
    for label, r in [("refractive", r_refr), ("non_refractive", r_nonrefr)]:
        rows_exp2.append({
            "test_depth_m": r["depth"],
            "model": label,
            "signed_mean_mm": r["signed_mean_mm"],
            "rmse_mm": r["rmse_mm"],
            "scale_factor": r["scale"],
            "calib_depth_min_m": EXP2_CALIB_RANGE[0],
            "calib_depth_max_m": EXP2_CALIB_RANGE[1],
        })
df_exp2 = pd.DataFrame(rows_exp2)

# --- Write CSVs ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
path_exp1 = OUTPUT_DIR / "exp1_parameter_errors.csv"
path_exp2 = OUTPUT_DIR / "exp2_depth_generalization.csv"
df_exp1.to_csv(path_exp1, index=False)
df_exp2.to_csv(path_exp2, index=False)

print(f"Wrote {path_exp1}  ({len(df_exp1)} rows)")
print(f"Wrote {path_exp2}  ({len(df_exp2)} rows)")

### Experiment 2 Takeaway

The refractive model's error stays low across all test depths — it learned the actual
geometry and can generalize. The non-refractive model's error grows as we move away
from the calibration depth, because refraction angle changes with depth and the wrong
model can't adapt without re-calibrating.

This is particularly important for applications where the measurement depth varies:
animal tracking, reef surveys, or any scenario where targets move in depth.

## Summary

Two experiments, one conclusion: **refraction is not a nuisance to be calibrated away;
it is a physical constraint that must be modeled explicitly.**

| Experiment | What we tested | Key finding |
|------------|---------------|-------------|
| 1 — Parameter Fidelity | Same data, two models | Non-refractive absorbs refraction into focal length and position bias |
| 2 — Depth Generalization | Calibrate narrow, test wide | Non-refractive error grows away from calibration depth |

**When is refractive calibration essential?**
- Camera-to-water distance is significant (≥ 0.3 m)
- Measurement depth varies across sessions or within a session
- High 3D accuracy is required (< 5 mm)
- You need stable calibration across depth ranges

**When might pinhole be acceptable?**
- Cameras are very close to the water surface (< 0.1 m)
- Only 2D detection or tracking is needed (no 3D reconstruction)
- Accuracy requirements are very relaxed (> 20 mm)
- Data collection depth is constant and pre-known

---

**Further reading:**
- [Full pipeline tutorial](01_full_pipeline.ipynb) — calibrate a rig end-to-end
- [Theory documentation](../guide/refractive_geometry.md) — refractive geometry model and Snell's law derivation
- [Optimizer guide](../guide/optimizer.md) — tuning the Stage 3 bundle adjustment